In [ ]:
import base64

run_id              = ""  # The actual RunID to substitute for '{run_id}' placeholders
sql_statements_b64  = ""  # base64-encoded SQL statements delimited by ";\n"

In [ ]:
import re

# Decode here (after Fabric/Papermill has injected the runtime parameters cell)
sql_statements = base64.b64decode(sql_statements_b64.encode()).decode() if sql_statements_b64 else ""

# ── Final hard gate before execution ───────────────────────────────────────────
# This is the chokepoint every caller (streamlit app or DE_NB_Main) passes through.
# Order: validate-all (no writes) -> baseline protection -> snapshot -> execute-all
# -> rollback on any failure. Conciseness rewrites are NOT done here — they happen
# upstream so the user approves exactly what runs.

ALLOWED_TABLES = {
    "input_cogs", "input_customer_capacities", "input_investor_capital",
    "input_performance_curves", "input_aggregated_raises", "input_aggregated_deployments",
    "input_fix_raises", "input_fix_deployments", "input_parameters",
    "input_investor_repayment_schedule", "input_net_new_capacities",
    "input_periods_config", "input_portfolios_config", "input_relative_deployments",
    "input_repeats_distribution", "input_time_periods",
}
ALLOWED_VERBS = {"UPDATE", "INSERT", "DELETE"}

statements = [s.strip() for s in sql_statements.split(";\n") if s.strip()]
print(f"Preparing {len(statements)} SQL statement(s) for RunID: {run_id}")


def _target_table(sql):
    m = re.match(r"\s*(?:UPDATE|DELETE\s+FROM|INSERT\s+INTO)\s+(\w+)", sql, re.IGNORECASE)
    return m.group(1) if m else None


# ── 1. Validate-all (no writes) ────────────────────────────────────────────────
resolved_statements, target_tables = [], []
for i, stmt in enumerate(statements, 1):
    upper = stmt.strip().upper()
    verb = upper.split()[0] if upper else ""
    assert verb in ALLOWED_VERBS, f"Statement {i}: disallowed verb '{verb}'"
    if verb in ("UPDATE", "DELETE"):
        assert re.search(r"WHERE\s+RUNID\s*=\s*'\{RUN_?ID\}'", upper), \
            f"Statement {i}: missing WHERE RunID = '{{run_id}}'"
    elif verb == "INSERT":
        assert "RUNID" in upper and "{RUN_ID}" in upper, \
            f"Statement {i}: INSERT must set RunID = '{{run_id}}'"

    table = _target_table(stmt)
    assert table and table.lower() in ALLOWED_TABLES, \
        f"Statement {i}: unknown or disallowed table in: {stmt[:80]}"

    # Substitute placeholder (handle both {run_id} and {RUN_ID} casings)
    resolved = stmt.replace("{run_id}", run_id).replace("{RUN_ID}", run_id)
    assert not re.search(r"\{\s*run_?id\s*\}", resolved, re.IGNORECASE), \
        f"Statement {i}: unresolved run_id placeholder remains after substitution"

    # Dry-run: validate table/columns/types are reachable via SELECT.
    # EXPLAIN is not supported for DML (UPDATE/DELETE) in Fabric Spark/Delta.
    where_m = re.search(r'\bWHERE\b(.+)$', resolved.strip(), re.IGNORECASE | re.DOTALL)
    where_clause = ("WHERE " + where_m.group(1).strip()) if where_m else ""
    spark.sql(f"SELECT COUNT(*) FROM {table} {where_clause}")

    resolved_statements.append(resolved)
    target_tables.append(table)

print("Validation passed for all statements.")

# ── 2. Baseline protection (authoritative — query the runs table) ──────────────
exists = spark.sql(f"SELECT COUNT(*) AS n FROM runs WHERE RunID = '{run_id}'").collect()[0]["n"]
assert exists > 0, f"RunID '{run_id}' not found in runs table — refusing to execute."
is_base = spark.sql(
    f"SELECT COUNT(*) AS n FROM runs WHERE RunID = '{run_id}' AND lower(Name) LIKE '%base%'"
).collect()[0]["n"]
assert is_base == 0, f"RunID '{run_id}' is a baseline run — refusing to mutate the baseline."
print("Baseline protection passed.")

# ── 3. Snapshot current Delta version of each distinct target table ────────────
# NOTE: rollback RESTOREs the whole table to this version. Runs are applied one at a
# time, so with no concurrent writer this reverts exactly the statements below.
snapshots = {}
for table in set(target_tables):
    try:
        snapshots[table] = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").collect()[0]["version"]
    except Exception as e:
        print(f"WARNING: no Delta history for {table} ({e}) — rollback unavailable for it.")

# ── 4. Execute-all, with 5. rollback on any failure ────────────────────────────
executed = 0
try:
    for i, resolved in enumerate(resolved_statements, 1):
        print(f"\n--- Statement {i} ---\n{resolved}\n")
        spark.sql(resolved)
        executed += 1
        print(f"Statement {i}: OK")
    print(f"\nAll {len(resolved_statements)} statement(s) executed successfully.")
except Exception as exec_err:
    print(f"\nERROR on statement {executed + 1}: {exec_err}\nRolling back…")
    for table, version in snapshots.items():
        try:
            spark.sql(f"RESTORE TABLE {table} TO VERSION AS OF {version}")
            print(f"  Restored {table} to version {version}.")
        except Exception as rb_err:
            print(f"  ROLLBACK FAILED for {table}: {rb_err}")
    raise

In [ ]:
mssparkutils.notebook.exit("ok")
